# Baseline Evaluation (Colab)

Colab port of `kaggle_baselines.ipynb`. Runs the physics and learned baselines against the IQ L=50 dataset; all baselines are compared on the same held-out test split.

**Difference from the Kaggle notebook:** the `.h5` and encoding `.sqlite3` are pulled from the Kaggle dataset `noso0s0n/iql50` via the Kaggle API (not mounted as a notebook input), and checkpoints + plots persist on a mounted Google Drive. Needs Colab Secrets `KAGGLE_USERNAME` and `KAGGLE_KEY` (from your `kaggle.json`).

**GPU:** every baseline runs on the GPU when one is present (the wrapper sets `DEVICE=cuda`, and `evaluate()` moves each batch there). The pairwise-sinc physics baselines (SAXS propoEst/stratEst especially) are O(atoms²) and were hours each on CPU; on a single T4 they are minutes. One GPU is enough, so on a 2x T4 runtime the second GPU stays idle.

**Metrics** (accumulated globally across the whole test set, not averaged per-batch; see `Baselines/metrics.py`):
- MSLE, R² (raw), R² (log1p), wall µs/atom — summary bar chart
- Per-q R², per-q %err, and an aggregated per-q summary — does performance hold at high q?
- Kratky log1p(I(q)) overlay — curve shape, not just scale
- **Per-molecule** signed-residual and MSLE distributions (with skew) — the honest error distribution, not the per-q mean proxy
- Error and signed-residual vs atom count (+ scaling-slope fit) — size-scaling and directional bias

**µs/atom caveat:** it is now wall-clock elapsed time per atom (`time.perf_counter`, bracketed with `torch.cuda.synchronize()` when a GPU is present), so it counts host and device work end to end. The old `time.process_time()` clock measured almost nothing once the heavy sinc sums moved to the GPU. Because it is wall-clock it depends on the machine, the device, and concurrent load, so only compare µs/atom between baselines from the same run on the same box. Do not compare across Kaggle and Colab, or across runs.

**Checkpointing / resume:** results are written to `CKPT_DIR` on Drive after every baseline. Rerun top-to-bottom to resume — completed baselines are skipped. Note the checkpointed baselines keep whatever µs/atom they were first measured with, so for one coherent wall-clock µs/atom column across all baselines, start from a fresh `CKPT_DIR` (a full GPU run is only minutes).

In [ ]:
# Data: the .h5 and encoding .sqlite3 are pulled from the Kaggle dataset
# noso0s0n/iql50. Drive is mounted only to persist checkpoints/plots.
# Auth: put your kaggle.json values in Colab Secrets (key icon) as
# KAGGLE_USERNAME and KAGGLE_KEY, with notebook access enabled.
import os, subprocess, sys

from google.colab import userdata

try:
    os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
    os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")
except Exception as e:
    raise RuntimeError(
        "Set Colab Secrets KAGGLE_USERNAME and KAGGLE_KEY from your kaggle.json."
    ) from e

KAGGLE_DATASET = "noso0s0n/iql50"
DATA_DIR = "/content/iql50"
HDF5_PATH = f"{DATA_DIR}/I(q)L50.h5"
DB_NAME = f"{DATA_DIR}/iq_train_set-ENCODING.sqlite3"

# pull only the two files we need (not the whole dataset / xyz coordinate files)
if not (os.path.exists(HDF5_PATH) and os.path.exists(DB_NAME)):
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "kaggle"], check=True
    )
    os.makedirs(DATA_DIR, exist_ok=True)
    for fname in ("I(q)L50.h5", "iq_train_set-ENCODING.sqlite3"):
        subprocess.run(
            [
                "kaggle",
                "datasets",
                "download",
                "-d",
                KAGGLE_DATASET,
                "-f",
                fname,
                "-p",
                DATA_DIR,
                "--unzip",
            ],
            check=True,
        )

# checkpoints + plots persist on Drive across sessions. The first mount call
# flakily raises even when it succeeds, so retry (force_remount on later tries).
from google.colab import drive

for _attempt in range(3):
    try:
        drive.mount("/content/drive", force_remount=_attempt > 0)
        break
    except Exception as _e:
        print(f"drive mount attempt {_attempt + 1} failed ({_e}); retrying...")
else:
    raise RuntimeError("Google Drive mount failed after 3 attempts")
CKPT_DIR = "/content/drive/MyDrive/APS360/baselines_ckpts"

REPO = "/content/APS360"
# How many of the 57 atom-size buckets to evaluate. 57 = all of them (test set
# stratified by size via the Batcher's 70/15/15 split). Lower only for speed:
# it keeps N_BUCKETS random buckets and drops the rest entirely.
N_BUCKETS = 57

assert os.path.exists(HDF5_PATH), f"HDF5 not found at {HDF5_PATH}"
assert os.path.exists(DB_NAME), f"encoding DB not found at {DB_NAME}"
os.makedirs(CKPT_DIR, exist_ok=True)
print("paths OK")

In [ ]:
import subprocess, sys

# install deps
%pip install -q xraydb beartype jaxtyping hdf5plugin h5py "scikit-learn>=1.3"

# plots (Baselines/metrics.py) render text through real LaTeX (xelatex) with
# JuliaMono — classic latex+dvipng can't load an arbitrary system font, only
# xelatex/lualatex + fontspec can. This apt step is the slow part (~2–3 min).
!sudo apt-get update -q && sudo apt-get install -y -q texlive-xetex texlive-latex-recommended texlive-fonts-recommended
!mkdir -p ~/.fonts && curl -fsSL https://github.com/cormullion/juliamono/releases/latest/download/JuliaMono-ttf.tar.gz \
    | tar -xz -C ~/.fonts && fc-cache -f ~/.fonts

# clone repo
if not os.path.exists(REPO):
    subprocess.run(
        ["git", "clone", "https://github.com/noshou/APS360.git", REPO],
        check=True,
    )
else:
    subprocess.run(["git", "-C", REPO, "pull"], check=True)

sys.path.insert(0, REPO)

In [ ]:
# ── checkpointing to the mounted Drive ── (plain file copy; no rclone needed) ─
import json
from Baselines.metrics import EvalResult

_CKPT = os.path.join(CKPT_DIR, "baselines_results.json")


def load_checkpoint() -> dict:
    """Return name -> EvalResult for baselines already completed on Drive.

    Returns the full result (per-q arrays + per-molecule distributions), so a
    resumed baseline still has plot data this session. Entries in the old
    pre-EvalResult schema (bare list) are dropped so they simply re-run.
    """
    if not os.path.exists(_CKPT):
        print("No existing checkpoint on Drive — fresh run.")
        return {}
    with open(_CKPT) as f:
        raw = json.load(f)
    data, stale = {}, []
    for name, v in raw.items():
        (
            data.__setitem__(name, EvalResult.from_json(v))
            if isinstance(v, dict)
            else stale.append(name)
        )
    if stale:
        print(
            f"Dropping {len(stale)} old-schema entry(ies), will re-run: {stale}"
        )
    print(f"Resumed {len(data)} completed baseline(s): {list(data.keys())}")
    return data


def save_checkpoint(results: dict) -> None:
    """Write the checkpoint to Drive. Call after every baseline.

    Writes to a temp file then renames, so a session that dies mid-write never
    leaves a truncated JSON that would fail to load on the next resume.
    """
    tmp = _CKPT + ".tmp"
    with open(tmp, "w") as f:
        json.dump(
            {name: r.to_json() for name, r in results.items()}, f, indent=2
        )
    os.replace(tmp, _CKPT)
    print(f"checkpoint saved to Drive ({len(results)} baseline(s))")

In [ ]:
import h5py, hdf5plugin, torch
from Preprocess.encode import Encoding
from ScatterNet.utils.config import DEFAULT_BUCKETS

print("Loading encoding DB...")
enc = Encoding(DB_NAME, HDF5_PATH)
print(f"  {enc.count():,} molecules  |  max atoms: {enc._max}")

with h5py.File(HDF5_PATH, "r") as f:
    q_grid = f["q_grid"][()]
    energy = float(f.attrs.get("energy", 10000.0))

q_grid = torch.from_numpy(q_grid).float()
print(f"  q_grid: {len(q_grid)} points  |  energy: {energy} eV")

In [ ]:
import random, time
from ScatterNet.batching import Batcher, Batch
from torch.utils.data import DataLoader

BUCKET_SAMPLE_SEED = 3092983  # deterministic; keep == kaggle_baselines.ipynb to sample the same buckets
LOADER_WORKERS = 0  # read in-process. workers>0 fork copies that each hold a
#   reference to the Batcher's big Python batch list; CPython
#   refcounting breaks copy-on-write and RAM climbs to an OOM
#   across a full pass. serial is slower but memory-stable.


def _first(x):
    return x[0]


class StreamingLoader:
    """Re-iterable, full-coverage view over a dataset split.

    Each iteration spins up a fresh DataLoader and yields batches one at a time,
    so only the batch in flight is ever resident -- never the whole split. Caching
    every batch instead exhausts /dev/shm at full coverage (the N_BUCKETS=57 OOM),
    and reading with worker processes leaks main RAM (copy-on-write on the big
    Python batch list); LOADER_WORKERS=0 avoids both. Trade-off: every .fit()/
    evaluate() pass re-reads the split from HDF5, so the run is slower but bounded
    in memory and resumable via the per-baseline checkpoints below.
    """

    def __init__(self, dataset, name, num_workers=LOADER_WORKERS):
        self.dataset = dataset
        self.name = name
        self.num_workers = num_workers

    def __len__(self):
        return len(self.dataset)

    def __iter__(self):
        loader = DataLoader(
            self.dataset,
            batch_size=1,
            collate_fn=_first,
            num_workers=self.num_workers,
        )
        t0, n = time.time(), len(self.dataset)
        for i, batch in enumerate(loader):
            if (i + 1) % 200 == 0 or (i + 1) == n:
                print(
                    f"\r  streaming {self.name}: {i + 1}/{n}  ({time.time() - t0:.0f}s)",
                    end="",
                    flush=True,
                )
            yield batch
        print()


eval_buckets = sorted(
    random.Random(BUCKET_SAMPLE_SEED).sample(
        DEFAULT_BUCKETS, min(N_BUCKETS, len(DEFAULT_BUCKETS))
    )
)

batcher = Batcher(
    hdf5_db=HDF5_PATH,
    enc=enc,
    batches=eval_buckets,
    seed=42,
    atom_size_ceil=6046,
)
train_set, _, test_set = batcher.get_sets()

# lazy: no HDF5 read happens here; each baseline streams the split when it runs
test_loader = StreamingLoader(test_set, "test set")
train_loader = StreamingLoader(train_set, "train set")
print(
    f"Test batches: {len(test_loader)}  |  Train batches: {len(train_loader)}"
)

In [ ]:
import torch
from Baselines.metrics import evaluate as _evaluate

# Run every baseline on the GPU when one is present (Colab / Kaggle 2x T4 -> cuda:0).
# The pairwise-sinc physics baselines (especially SAXS propoEst / stratEst) are
# O(m^2) and were ~11 h each on CPU; on a single T4 they drop to minutes. evaluate()
# moves each batch to DEVICE and every baseline is device-following, so this is all
# that is needed. One GPU is far more than enough for this workload, so we use cuda:0
# and leave the second T4 idle rather than add multi-GPU sharding to a run that is
# already only minutes long.
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Baselines device: {DEVICE}")


def evaluate(baseline, loader, name):
    """Evaluate a baseline, print its headline metrics, return the full EvalResult."""
    result = _evaluate(baseline, loader, q_grid, name, device=DEVICE)
    print(
        f"{name:<30s}  MSLE={result.msle:.4f}  R²(raw)={result.r2_raw:.4f}  "
        f"R²(log1p)={result.r2_log1p:.4f}  {result.us_per_atom:.2f} μs/atom"
    )
    return result

In [ ]:
sys.path.insert(0, f"{REPO}/Baselines/physics-benchmarks")
sys.path.insert(0, f"{REPO}/Baselines/learned-benchmarks")

from rg import RgBaseline, GuinierPorodBaseline
from atom_count import AtomCountBaseline
from binned_debye import BinnedDebyeBaseline
from propo_est import PropoEstBaseline
from strat_est import StratEstBaseline

print("=== Physics Baselines ===")
results = load_checkpoint()

# factories, not instances: construction/fit is deferred until we know a
# baseline isn't already in `results`, so a completed one never re-fits on resume
physics_baselines = [
    ("Guinier (Rg)", lambda: RgBaseline(q_grid, energy)),
    ("Guinier-Porod", lambda: GuinierPorodBaseline(q_grid, energy)),
    ("Atom Count", lambda: AtomCountBaseline().fit(train_loader)),
    ("Binned Debye", lambda: BinnedDebyeBaseline(q_grid, energy)),
    ("SAXS propoEst", lambda: PropoEstBaseline(q_grid, energy, e=0.380)),
    ("SAXS stratEst", lambda: StratEstBaseline(q_grid, energy, a=0.6)),
]

for name, make_baseline in physics_baselines:
    if name in results:
        print(f"{name:<30s}  (skipped, resumed from checkpoint)")
        continue
    results[name] = evaluate(make_baseline(), test_loader, name)
    save_checkpoint(results)

In [ ]:
# self-contained: put the learned-benchmarks dir on sys.path here too, so this
# cell runs even when the physics cell above hasn't this session (fresh runtime,
# or run-from-here). Idempotent: a duplicate leading path entry is harmless.
sys.path.insert(0, f"{REPO}/Baselines/learned-benchmarks")

from mlp_2 import Mlp2Baseline
from linsvm import Linsvm
from nearest_neighbour import NNBaseline

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
print("=== Learned Baselines ===")

learned_baselines = [
    ("MLP", lambda: Mlp2Baseline().fit(train_loader)),
    ("Linear SVM", lambda: Linsvm().fit(train_loader)),
    (
        "Nearest Neighbour",
        lambda: NNBaseline(q_grid, energy).fit(train_loader),
    ),
]

for name, make_baseline in learned_baselines:
    if name in results:
        print(f"{name:<30s}  (skipped, resumed from checkpoint)")
        continue
    results[name] = evaluate(make_baseline(), test_loader, name)
    save_checkpoint(results)

In [ ]:
_known = {n for n, _ in physics_baselines} | {n for n, _ in learned_baselines}
_stale = sorted(n for n in results if n not in _known)
if _stale:
    print(
        f"Dropping stale checkpoint entries no longer in the baseline list: {_stale}"
    )
    for n in _stale:
        del results[n]
    save_checkpoint(results)

print("\n=== Summary ===")
print(
    f"{'Baseline':<30s}  {'MSLE':>8s}  {'R²(raw)':>10s}  {'R²(log1p)':>12s}  {'μs/atom':>10s}  {'resid skew':>11s}"
)
print("-" * 90)
for name, r in sorted(results.items(), key=lambda x: x[1].msle):
    skew = (r.resid_stats or {}).get("skew", float("nan"))
    print(
        f"{name:<30s}  {r.msle:>8.4f}  {r.r2_raw:>10.4f}  {r.r2_log1p:>12.4f}  {r.us_per_atom:>10.2f}  {skew:>11.3f}"
    )

In [ ]:
from Baselines.metrics import run_all_plots

PLOTS_DIR = os.path.join(CKPT_DIR, "baseline_plots")
written = run_all_plots(list(results.values()), q_grid, PLOTS_DIR)
print(
    f"wrote {len(written)} plot(s) to Drive:", PLOTS_DIR, *written, sep="\n  "
)

Plots written to `CKPT_DIR/baseline_plots` on Drive: `summary` and `per_q_summary` (each as both a bar-chart `.png` and a `.json` with the same underlying numbers, so the raw values survive independent of the image), `per_q_r2`, `per_q_percent_error`, `error_vs_atom_count`, `residual_vs_atom_count`, plus per-baseline `kratky_*`, `residual_histogram_*` (per-molecule signed residual + skew, reported in the legend rather than a title), and `per_molecule_msle_*`. See `Baselines/metrics.py` for what each shows.